# Welcome to Jupyter Notebooks: Your Interactive Python Playground

Think of Jupyter as a supercharged calculator meets a lab notebook. Unlike traditional Python scripts that run top-to-bottom once, Jupyter lets you experiment, iterate, and explore data interactively. This is **the** tool for data science, quant research, and rapid prototyping.

**Prework.** Run through the first plot and practice Restart & Run All.
 The API and magic-command sections are optional reference. Continue with [Pandas to portfolio](2_pandas_to_portfolio.ipynb).
 All default examples run offline; set `USE_LIVE_API = True` only if you want to use your own API key.

## The Building Block: Cells

Everything in Jupyter happens in **cells**. Each cell is an independent unit you can run, edit, and re-run. There are two main types:
- **Code cells**: Execute Python code
- **Markdown cells**: Format text, equations, images (like this one!)

### Two Modes: Edit vs Command
- **Edit mode** (green border): You're typing inside a cell
- **Command mode** (blue border): You're navigating between cells

**Try this:** Click on the cell below, then press `Esc` to enter command mode. See the border turn blue?

In [ ]:
# Click me, then press Esc to see command mode
# Press Enter to come back to edit mode
print("I'm learning Jupyter!")

### Essential Keyboard Shortcuts

Master these and you'll fly through your analysis:

**In Command Mode:**
- `A` - Insert cell **A**bove
- `B` - Insert cell **B**elow
- `DD` - **D**elete cell (press D twice)
- `C` - **C**opy cell
- `V` - Paste cell (think **V** for paste)
- `M` - Convert to **M**arkdown
- `Y` - Convert to code (p**Y**thon)

**In Either Mode:**
- `Shift + Enter` - Run cell and move to next (most common!)
- `Ctrl + Enter` - Run cell and stay put (use for iterating)

**Let's practice:** Try copying the cell below and pasting it underneath!

In [ ]:
# Select this cell in command mode (press Esc), press C to copy, then V to paste
experiment_number = 1
print(f"This is experiment #{experiment_number}")

## The Secret Sauce: Non-Linear Execution

Here's what makes Jupyter special: **you can run cells in any order**. Notice the `[ ]` brackets to the left of each code cell? After running, you'll see numbers like `[1]`, `[2]`, etc. These track execution order.

**This is powerful but dangerous!** Let's see why...

In [ ]:
# Run me FIRST
portfolio_value = 1_000_000
print(f"Starting portfolio: ${portfolio_value:,}")

In [ ]:
# Run me SECOND - I depend on the cell above!
daily_return = 0.02  # 2% gain
portfolio_value *= (1 + daily_return)
print(f"After one day: ${portfolio_value:,}")

In [ ]:
# Now run me multiple times - watch what happens!
# Each run compounds the return again
portfolio_value *= (1 + daily_return)
print(f"Current value: ${portfolio_value:,.2f}")

**See what happened?** The execution numbers `[1]`, `[2]`, `[3]`... show you the order. If you ran the last cell multiple times, your portfolio kept growing!

**Pro tip:** When things get messy, go to `Kernel > Restart & Run All` to reset and run everything top-to-bottom.

## Real Work: Import Libraries & Visualize Data

Let's do something practical. Jupyter really shines when you're exploring data and creating visualizations. We'll create a simple financial dataset and plot it.

In [ ]:
# Standard imports for data analysis
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# This magic command makes plots appear inline
%matplotlib inline

print("Libraries loaded! Ready to analyze data.")

Now watch this - we'll create data in one cell and visualize it in another. **This is the interactive workflow in action!**

In [ ]:
# Simulate a stock price with drift and volatility
np.random.seed(42)  # For reproducibility
days = 252  # Trading days in a year
returns = np.random.randn(days) * 0.03 + 0.0005  # 3% daily vol, 0.05% drift
price = 100 * np.exp(np.cumsum(returns))  # Geometric Brownian motion

df = pd.DataFrame({
    'Day': range(1, days + 1),
    'Price': price
})

print(f"Created {len(df)} days of price data")
df.head()  # Jupyter automatically displays the DataFrame nicely!

In [ ]:
# Plot the data - see how easy this is?
df.set_index('Day')['Price'].plot(figsize=(12, 6), title='Simulated Stock Price Over One Year')
plt.ylabel('Price ($)')
plt.xlabel('Trading Day')
plt.grid(alpha=0.3)
plt.show()

**Now for the magic:** Go back to the cell where we created `df` and change the `seed` value or the volatility (`0.03`). Run that cell, then run the plotting cell again. **You just iterated on your analysis without re-running everything!** This is why data scientists love Jupyter.

## Getting Real Market Data

Let's pull actual stock data from an API. This demonstrates how you'd typically start a quant analysis in Jupyter: get data, explore it, visualize it, iterate.

The bundled historical sample lets you run this lesson without credentials. Set `AV_API_KEY` in your local environment to try the live API instead. The sample uses AAPL prices from the course database, formatted like an Alpha Vantage response; it is not a live API response.

In [ ]:
import requests
from dotenv import load_dotenv
import os

# Load API keys from .env file (never hardcode secrets!)
load_dotenv()
import json
from pathlib import Path


In [ ]:
# Fetch Apple stock data, or use the bundled historical sample without an API key.
symbol = 'AAPL'  # With an API key, try 'NVDA' or 'MSFT'.
api_key = os.getenv('AV_API_KEY')

USE_LIVE_API = False

if USE_LIVE_API:
    if not api_key:
        raise ValueError('Set AV_API_KEY locally before opting in to live data.')
    response = requests.get(
        'https://www.alphavantage.co/query',
        params={
            'function': 'TIME_SERIES_DAILY',
            'symbol': symbol,
            'apikey': api_key,
        },
        timeout=30,
    )
    response.raise_for_status()
    data = response.json()
    if 'Time Series (Daily)' not in data:
        raise ValueError('No daily prices returned. Check the API key and request quota.')
    print(f'Fetched live data for {symbol}; status: {response.status_code}')
else:
    symbol = 'AAPL'  # The bundled sample contains only AAPL.
    data = json.loads((next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file()) / 'Lectures/Lecture 2/data/aapl_sample.json').read_text())
    print('Using the bundled historical AAPL sample (not live market data).')


In [ ]:
# Let's peek at the raw data structure
print("Keys in response:", list(data.keys()))
print("\nFirst date in data:", list(data['Time Series (Daily)'].keys())[0])
print("First date's values:", list(data['Time Series (Daily)'].values())[0])

In [ ]:
# Transform JSON into a clean DataFrame
df = pd.DataFrame.from_dict(
    data['Time Series (Daily)'],
    orient='index'
)

# Clean up: rename columns and fix data types
df.columns = ['open', 'high', 'low', 'close', 'volume']
df.index = pd.to_datetime(df.index)
df.index.name = 'date'

# Convert strings to numbers
for col in df.columns:
    df[col] = pd.to_numeric(df[col])

# Sort by date (oldest first)
df = df.sort_index()

print(f"Data shape: {df.shape}")
df.tail()  # Show most recent data

In [ ]:
# Visualize the closing price
fig, ax = plt.subplots(figsize=(14, 7))
df['close'].plot(ax=ax, linewidth=2)
ax.set_title(f'{symbol} Closing Price', fontsize=16, fontweight='bold')
ax.set_ylabel('Price ($)', fontsize=12)
ax.set_xlabel('Date', fontsize=12)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Try this now:**
1. Set your own `AV_API_KEY` and go back to the cell where we set `symbol = 'AAPL'`
2. Change it to `symbol = 'NVDA'` or `symbol = 'MSFT'`
3. Re-run just that cell and the cells below it
4. Watch how quickly you can compare different stocks!

**This is the Jupyter workflow:** Fetch data once, then iterate rapidly on your analysis.

## Jupyter Magic Commands: Superpowers Unlocked

Jupyter has special commands called "magics" that start with `%` (line magic) or `%%` (cell magic). They're not Python - they're Jupyter-specific tools.

### 1. Timing Code Performance

When optimizing your code, you need to measure it. Jupyter makes this trivial:

In [ ]:
# Time a single line with %time (runs once)
%time result = sum([i**2 for i in range(1000)])

In [ ]:
%%timeit
# Time an entire cell with %%timeit (runs many times for accuracy)
result = sum([i**2 for i in range(1000)])

In [ ]:
%%timeit
# Compare with numpy - which is faster?
result = np.sum(np.arange(1000)**2)

### 2. Environment Variables

Quick access to your environment (useful for debugging API keys, paths, etc.):

In [ ]:
# Set an environment variable
%env TEST_PORTFOLIO_VALUE=1000000

In [ ]:
# Access it from Python
portfolio = int(os.environ['TEST_PORTFOLIO_VALUE'])
print(f"Portfolio value: ${portfolio:,}")

In [ ]:
# See all environment variables (output truncated for readability)
env_vars = %env
print(f"Total environment variables: {len(env_vars)}")
print("\nSome important ones:")
for key in ['PATH', 'HOME', 'USER']:
    if key in env_vars:
        print(f"{key}: {env_vars[key]}")

### 3. Other Useful Magics

Here are more tricks you'll use often:

In [ ]:
# List all variables in your namespace
%who

In [ ]:
# Inspect a specific safe variable rather than dumping credentials from the namespace.
print(df.shape)

In [ ]:
# Run a Python script from your notebook
# %run my_script.py

# Load code from an external file into a cell
# %load my_script.py

# See all available magic commands
%lsmagic

## Pro Tips for Effective Jupyter Usage

### 1. Keep Your Kernel Clean
- When things get confusing: `Kernel > Restart & Clear Output`
- To verify everything works: `Kernel > Restart & Run All`
- If code behaves weirdly, check those execution numbers `[1]`, `[2]` - did you run cells out of order?

### 2. Document As You Go
- Use markdown cells liberally to explain your thinking
- Future you (and your teammates) will thank you
- For homework: good documentation = easier grading = better feedback

### 3. Iterative Development
- Start with small experiments in individual cells
- Once code works, you can combine cells or refactor into functions
- Don't be afraid to create throwaway cells for testing

### 4. Master the Shortcuts
- `Shift + Enter` - Run and advance (use this 90% of the time)
- `Ctrl + Enter` - Run and stay (great for tweaking code)
- `Esc` then `A` or `B` - Add cells above/below
- `Esc` then `DD` - Delete cell

### 5. Notebook != Production Code
- Notebooks are for exploration and analysis
- Once your logic is solid, refactor into `.py` modules
- Think of notebooks as your lab notebook, not your final paper

## Wrap-Up

You now understand the core concepts that make Jupyter powerful:

- **Cells** are your execution units - code or markdown
- **Non-linear execution** lets you iterate rapidly but requires discipline
- **Interactive visualization** makes data exploration intuitive
- **Magic commands** provide quick access to timing, environment, and more
- **Keyboard shortcuts** make you faster and more efficient

Throughout this course, you'll use Jupyter for homework, data exploration, and building models. The more comfortable you get with the interactive workflow, the more productive you'll be.

**Next steps:**
- Experiment with the examples above
- Try modifying parameters and re-running cells
- Get comfortable with the keyboard shortcuts
- Don't be afraid to break things - just restart the kernel!

Happy exploring!